# Expanded_Eurofer_CD â€” Cluster Dynamics for bcc Fe / EUROFER97

Full per-size cluster dynamics model implementing:
- **Master equations** (Eqs. 152, 155, 157) for SIA clusters, He-vacancy bubbles, and free He.
- **He-reduction**: Case 2 fission (Eq. 175, decoupled) or Case 1 fusion (Eq. 174, mean-field).
- **Size-bin moments** (Chapter 9, Eqs. 188â€“211) for large-N efficiency.
- **Solute trapping** in EUROFER97 (Cr, W, Mn, C, N) via effective diffusivities (Eqs. 42, 48, 52).
- **1D/3D mixed transport** for glissile SIA clusters (Eq. 141).

Physics reference: Ghoniem, N.M. (2026), *'A Cluster Dynamics Model for Radiation Damage Evolution in Ferritic-Martensitic Steels'* (Rate_Equations.pdf).

---

## Solver modes
| `solver_mode` | Description |
|---|---|
| `cpp_full` | C++ CVODE BDF, full system, dense/band/GMRES |
| `cpp_sliding_win` | C++ CVODE BDF with sliding SIA window (Phase III) |
| `sliding_OpenMP` | Sliding window + OpenMP intra-RHS parallelism (Phase IV) |

## Physics options
| `physics_option` | Equations | He reduction | Cascade |
|---|---|---|---|
| `full_CD_fission` | Eqs. 152, 155, 157 | Case 2, Eq. 175 (decoupled) | Fission (Table 2) |
| `full_CD_fusion` | Eqs. 152, 155, 157 | Case 1, Eq. 174 (mean-field) | Fusion (Table 2) |
| `bin_moment_CD_fission` | Chapter 9 moments | Case 2 | Fission |
| `bin_moment_CD_fusion` | Chapter 9 moments | Case 1 | Fusion |

## Build C++ solver (run once after code changes)

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Add Expanded_Eurofer_CD root to path
MODULE_ROOT = Path('..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

print(f'Module root: {MODULE_ROOT}')
print(f'Repo root:   {REPO_ROOT}')


# Build the C++ SUNDIALS solver
build_dir = MODULE_ROOT / 'build'
build_dir.mkdir(exist_ok=True)

cmake_src  = MODULE_ROOT / 'cpp_utils'
cmake_cmd  = ['cmake', '-S', str(cmake_src), '-B', str(build_dir),
               '-DCMAKE_BUILD_TYPE=Release']
build_cmd  = ['cmake', '--build', str(build_dir), '--config', 'Release']

for cmd in [cmake_cmd, build_cmd]:
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(' '.join(cmd))
    if res.returncode != 0:
        print('STDERR:', res.stderr[-2000:])
    else:
        print('OK')
        print(res.stdout[-500:])

## MAIN SIMULATION SCRIPT

### Bin-moment system size (N_eq)

The total number of ODEs depends on `I`, `V`, `i_discrete`, `I_bin`, `V_bin`, and the He mode:

```
i_discrete = max discrete SIA size (individually tracked)
v_discrete = max discrete vacancy size (individually tracked)
I_bin      = number of SIA bin-moment equations beyond i_discrete
V_bin      = number of VAC bin-moment equations beyond v_discrete
```

| Component | ODEs | Description |
|---|---|---|
| Discrete SIA | i_discrete | individual c_i for i=1..i_discrete |
| Binned SIA moments | 2 × I_bin | zeroth + first moment per bin |
| Discrete VAC | v_discrete | individual c_v for v=1..v_discrete |
| Binned VAC moments | 2 × V_bin | zeroth + first moment per bin (hat-function closure) |
| He state | 1 (fission/Case 2) or V_bin (fusion/Case 1) | Q_tot scalar or Q_k per bin |
| Free He | 0 (QSS) or 1 (dynamic) | quasi-steady-state eliminates this |
| **Total N_eq** | **i_discrete + 2·I_bin + v_discrete + 2·V_bin + 1** (fission, QSS) | |

**Limiting cases:**
- When `I_bin=0` and `i_discrete=I` → all SIA equations are discrete (full_CD recovery)
- When `V_bin=0` and `v_discrete=V` → all VAC equations are discrete

**Examples** (fission, QSS He, i_discrete=i_mobile=11, v_discrete=v_mobile=5):

| I | V | I_bin | V_bin | N_eq |
|---|---|---|---|---|
| 10,000 | 10,000 | 10 | 13 | 63 |
| 100,000 | 100,000 | 13 | 16 | 74 |
| 1,000,000 | 1,000,000 | 17 | 19 | 88 |

Compare with full_CD: N_eq = I + V + 2 (fission), so I=V=10,000 → **20,002 ODEs**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, io
from pathlib import Path

MODULE_ROOT = Path('..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import numpy as np
import importlib

# Reload all py_utils modules to pick up code changes
import Expanded_Eurofer_CD.py_utils.defect_production as _dp_mod
import Expanded_Eurofer_CD.py_utils.binding_energies as _be_mod
import Expanded_Eurofer_CD.py_utils.bin_moment_rates as _bmr
import Expanded_Eurofer_CD.py_utils.input_data as _inp_mod
import Expanded_Eurofer_CD.py_utils.reaction_rates as _rr_mod
import Expanded_Eurofer_CD.py_utils.rate_equations as _re_mod
import Expanded_Eurofer_CD.py_utils.cpp_bridge as _cb_mod
import Expanded_Eurofer_CD.py_utils.post_process as _pp_mod
import Expanded_Eurofer_CD.py_utils.simulation as _sim_mod
import Expanded_Eurofer_CD.py_utils.visualization as viz
for _m in [_dp_mod, _be_mod, _bmr, _inp_mod, _rr_mod, _re_mod,
           _cb_mod, _pp_mod, _sim_mod, viz]:
    importlib.reload(_m)
from Expanded_Eurofer_CD.py_utils.simulation import ExpandedEuroferCDSimulation


# ══════════════════════════════════════════════════════════════════════════════
# 2. User selections — cluster domain and mobility
# ══════════════════════════════════════════════════════════════════════════════
DEBUG = False
SOLVER_MODE    = 'cpp_full'
PHYSICS_OPTION = 'bin_moment_CD_fission'

# ── Domain sizes ─────────────────────────────────────────────────────────────
I          = int(1e4)    # max SIA cluster size  (grows adaptively if doublings > 0)
V          = int(1e4)    # max vacancy cluster size

# ── Mobility cutoffs ────────────────────────────────────────────────────────
i_mobile   = 10          # max mobile SIA cluster size
v_mobile   = 5          # max mobile vacancy cluster size

# ── Discrete / binned split ──────────────────────────────────────────────────
# Sizes 1..i_discrete tracked as individual ODEs; beyond that, bin moments.
# Defaults: i_discrete = i_mobile, v_discrete = v_mobile.
# Set I_bin = 0 and i_discrete = I to recover full_CD limit.
i_discrete = 100    # max discrete SIA size
v_discrete = 100    # max discrete vacancy size
I_bin      = 5         # SIA bin-moment equations (None = auto from r≈2)
V_bin      = 10         # VAC bin-moment equations (None = auto from r≈2)

# ── Other settings ───────────────────────────────────────────────────────────
C_FLOOR    = 1e-25       # concentration floor [atom fraction]
HE_OPTIONS = 'quasi_steady_state'


# ══════════════════════════════════════════════════════════════════════════════
# 3. Solver configuration


# ══════════════════════════════════════════════════════════════════════════════
SOLVER_CONFIG = {
    't_span':   (1e-6, 1e6),
    'n_points': 200,
    'log_time': True,
    'rtol':     1e-6,
    'atol':     1e-25,
    'solver_method': {
        'backend':           'cvode',
        'lmm':               'bdf',
        'linsol':            'gmres',
        'window_w0_i':       50,
        'window_width':      200,
        'window_C_expand':   1e-18,
        'window_expand_pad': 10,
        'window_omp_threads':0,
        'window_prec':       1,
        'window_gmres_maxl': 20,
        'window_N_thresh':   500,
    }
}


# ══════════════════════════════════════════════════════════════════════════════
# 4. Initialise simulation (reads Excel file)
# ══════════════════════════════════════════════════════════════════════════════

_saved_out, _saved_err = sys.stdout, sys.stderr
if not DEBUG:
    sys.stdout = sys.stderr = io.StringIO()
try:
    sim = ExpandedEuroferCDSimulation(
        I=I, V=V,
        solver_mode=SOLVER_MODE,
        physics_option=PHYSICS_OPTION,
        C_floor=C_FLOOR,
        he_options=HE_OPTIONS,
        i_mobile=i_mobile,
        v_mobile=v_mobile,
    )
finally:
    sys.stdout, sys.stderr = _saved_out, _saved_err


# ══════════════════════════════════════════════════════════════════════════════
# 4b. Parameter overrides — applied AFTER reading Excel, BEFORE solver run.
#     Keys are the Symbol names from each Excel sheet.
#     Set PARAM_OVERRIDES = {} to use the Excel defaults.
# ══════════════════════════════════════════════════════════════════════════════

PARAM_OVERRIDES = {
    # ── Production sheet (fission) ────────────────────────────────────────
    'eta':       0.3,        # cascade survival efficiency (default 0.30)
    'f_cl_i':    0.58,       # SIA clustering fraction    (default 0.58)
    'f_cl_v':    0.45,       # vacancy clustering fraction (default 0.15)

    # ── Diffusion sheet ──────────────────────────────────────────────────
    'E_m_1D':    0.34,       # 1D SIA cluster migration energy [eV] (default 0.34)
    'i_mobile':  i_mobile,   # SIA mobility cutoff
    'L_hat':     71.8,       # mean free path L/a (default 50)
    'c_C':       1.94e-4,    # dissolved C concentration [at/at] (default 5e-4)
    'E_b_C_SIA': 0.45,       # C-SIA binding energy [eV] (default 0.45)

    # ── Reactions sheet ──────────────────────────────────────────────────
    'rho_d':     1e14,       # dislocation density [m^-2] (default 1e13)
    'Z_i':       1.1,        # SIA dislocation bias factor (default 1.1)
    'Z_ii':      1.1,        # SIA-SIA coalescence bias factor (default 1.0)
    'shape_function': 'linear',  # 'constant', 'linear', or 'lognormal'
   # 'boundary_flux': 'reflection',   # or 'absorption' (default)
}

# Inject discrete/bin controls into overrides
PARAM_OVERRIDES['i_discrete'] = i_discrete
PARAM_OVERRIDES['v_discrete'] = v_discrete
if I_bin is not None:
    PARAM_OVERRIDES['I_bin'] = I_bin
if V_bin is not None:
    PARAM_OVERRIDES['V_bin'] = V_bin

if PARAM_OVERRIDES:
    _saved_out2, _saved_err2 = sys.stdout, sys.stderr
    if not DEBUG:
        sys.stdout = sys.stderr = io.StringIO()
    try:
        inp = sim.input_data
        for key, val in PARAM_OVERRIDES.items():
            placed = False
            for d in [inp.production_fission, inp.production_fusion,
                      inp.diffusion, inp.reactions,
                      inp.energetics, inp.dissociation]:
                if key in d:
                    d[key] = val
                    placed = True
            if not placed:
                inp.reactions[key] = val
        if 'i_mobile' in PARAM_OVERRIDES:
            inp.diffusion['i_mobile'] = int(PARAM_OVERRIDES['i_mobile'])
            inp.reactions['i_mobile'] = int(PARAM_OVERRIDES['i_mobile'])
        inp._calculate_derived()
        sim.rebuild_rates()
    finally:
        sys.stdout, sys.stderr = _saved_out2, _saved_err2

    print(f'Applied {len(PARAM_OVERRIDES)} parameter overrides:')
    for k, v in PARAM_OVERRIDES.items():
        print(f'  {k:>12} = {v}')
else:
    print('Using Excel defaults (no overrides)')

# ── Report system size ────────────────────────────────────────────────────
re = sim.rate_equations
if hasattr(re, 'I_bin'):
    P = re.n_mom
    print(f'\nHybrid discrete + bin-moment system (shape_function={re.shape_function!r}):')
    print(f'  Discrete SIA:  i = 1..{re.i_discrete}  ({re.i_discrete} ODEs)')
    print(f'  Binned SIA:    I_bin = {re.I_bin}  ({P} moments each = {P*re.I_bin} ODEs)')
    print(f'  Discrete VAC:  v = 1..{re.v_discrete}  ({re.v_discrete} ODEs)')
    print(f'  Binned VAC:    V_bin = {re.V_bin}  ({P} moments each = {P*re.V_bin} ODEs)')
    he_odes = re.N_eq - (re.i_discrete + P*re.I_bin + re.v_discrete + P*re.V_bin)
    print(f'  He state:      {he_odes} ODE(s)')
    print(f'  ──────────────────────')
    print(f'  Total N_eq = {re.N_eq}  (full_CD would be {sim.input_data.I + sim.input_data.V + 2})')
else:
    print(f'\nFull per-size system: N_eq = {re.N_eq}')


# ══════════════════════════════════════════════════════════════════════════════
# 5. Build live progress callback
# ══════════════════════════════════════════════════════════════════════════════
sim._progress_rows = []

rr      = sim.reaction_rates
rate_eq = sim.rate_equations
G_dpa   = sim.input_data.derived['G']
Omega   = sim.input_data.derived['Omega']
s2m     = 1.0 / Omega

_row_idx = [0]

def _progress_callback(row):
    sim._progress_rows.append(dict(row))
    j   = _row_idx[0];  _row_idx[0] += 1
    t_  = row.get('t', 0.0)
    dos = t_ * G_dpa
    if DEBUG:
        ci1 = row.get('c_i1', 0.0)
        cv1 = row.get('c_v1', 0.0)
        sys.stderr.write(
            f'  [{j:>4d}] t={t_:.4e}  dose={dos:.3e}'
            f'  Ci1={ci1*s2m:.3e}  Cv1={cv1*s2m:.3e} m^-3\n')
        sys.stderr.flush()


# ══════════════════════════════════════════════════════════════════════════════
# 6. Run simulation — TRULY ADAPTIVE CONTINUATION
#    Integration runs in short segments (points_per_segment output points
#    each).  After every segment the tail fraction is checked.  When the
#    threshold is first exceeded, I and/or V are doubled and
#    integration CONTINUES from that time point — no restart from t=0.
#    I and V have independent doubling budgets.
# ══════════════════════════════════════════════════════════════════════════════
%matplotlib inline

results = sim.run_adaptive(
    solver_config=SOLVER_CONFIG,
    save_output=True,
    progress_callback=_progress_callback,
    boundary_threshold=0.1,     # adapt if >10% of content at boundary
    max_doublings=0,            # 0 = no adaptive doubling
    points_per_segment=10,       # check every 10 output points
)

if results is not None:
    t_arr = results['t']
    I_final = sim.input_data.I
    V_final = sim.input_data.V
    print(f'\nFinal domain: I={I_final}  V={V_final}')
    print(f'Solution:         {len(t_arr)} time points, '
          f't = [{t_arr[0]:.2e}, {t_arr[-1]:.2e}] s')
    print(f'Final dose:       {results["dose"][-1]:.4e} dpa')
    print(f'Swelling (final): {results["swelling"][-1]*100:.6f} %')
    print(f'C_He_tot (final): {results["C_He_tot"][-1]:.3e} m^-3')
    print(f'mean_n_i (final): {results["mean_n_i"][-1]:.2f}')
    print(f'mean_n_v (final): {results["mean_n_v"][-1]:.2f}')
    print(f'N_loops  (final): {results["N_loops"][-1]:.3e} m^-3')
    print(f'N_voids  (final): {results["N_voids"][-1]:.3e} m^-3')
    print(f'delta_FP (final): {results["delta_FP"][-1]:.3e}  (Eq. 164)')
    print(f'delta_He (final): {results["delta_He"][-1]:.3e}  (Eq. 165)')
else:
    print('Simulation failed -- check C++ build and parameter file.')

Applied 14 parameter overrides:
           eta = 0.3
        f_cl_i = 0.58
        f_cl_v = 0.45
        E_m_1D = 0.34
      i_mobile = 10
         L_hat = 71.8
           c_C = 0.000194
     E_b_C_SIA = 0.45
         rho_d = 100000000000000.0
           Z_i = 1.1
          Z_ii = 1.1
  shape_function = linear
    i_discrete = 10000
    v_discrete = 10000

Hybrid discrete + bin-moment system (shape_function='linear'):
  Discrete SIA:  i = 1..10000  (10000 ODEs)
  Binned SIA:    I_bin = 0  (2 moments each = 0 ODEs)
  Discrete VAC:  v = 1..10000  (10000 ODEs)
  Binned VAC:    V_bin = 0  (2 moments each = 0 ODEs)
  He state:      6 ODE(s)
  ──────────────────────
  Total N_eq = 20006  (full_CD would be 20002)

[Segment 1] I=10000  V=10000  t=[1.00e-06, 3.98e-06]

Launching solver_mode='cpp_full' …
C++ solver: solver.exe  N_eq=20006  solver_mode='cpp_full'  physics='bin_moment_CD_fission'


Expanded_Eurofer_CD solver: N_eq=20006  physics_option=2  he_mode=0  he_options=quasi_steady_state  C_floor=1e-25  window_mode=0
[cvode] pt=1/10  t=1.16591e-06  steps=18  rhs=23  nlcf=0  etf=0  h=6.91266e-08  ret=0
[cvode] pt=2/10  t=1.359e-06  steps=21  rhs=26  nlcf=0  etf=0  h=1.433e-07  ret=0
[cvode] pt=3/10  t=1.585e-06  steps=22  rhs=27  nlcf=0  etf=0  h=3.244e-07  ret=0
[cvode] pt=4/10  t=1.848e-06  steps=23  rhs=28  nlcf=0  etf=0  h=1.139e-06  ret=0
[cvode] pt=5/10  t=2.154e-06  steps=23  rhs=28  nlcf=0  etf=0  h=1.139e-06  ret=0
[cvode] pt=6/10  t=2.512e-06  steps=23  rhs=28  nlcf=0  etf=0  h=1.139e-06  ret=0
[cvode] pt=7/10  t=2.929e-06  steps=23  rhs=28  nlcf=0  etf=0  h=1.139e-06  ret=0
[cvode] pt=8/10  t=3.415e-06  steps=24  rhs=30  nlcf=0  etf=0  h=1.139e-06  ret=0
[cvode] pt=9/10  t=3.981e-06  steps=24  rhs=30  nlcf=0  etf=0  h=1.139e-06  ret=0
Done: 10 time points written.


C++ solver completed — 10 time points
Results processing complete.
  Tail OK: SIA=0.000  VAC=0.000

[Segment 2] I=10000  V=10000  t=[3.98e-06, 1.58e-05]

Launching solver_mode='cpp_full' …
C++ solver: solver.exe  N_eq=20006  solver_mode='cpp_full'  physics='bin_moment_CD_fission'


Expanded_Eurofer_CD solver: N_eq=20006  physics_option=2  he_mode=0  he_options=quasi_steady_state  C_floor=1e-25  window_mode=0
[cvode] pt=1/10  t=4.64159e-06  steps=10  rhs=18  nlcf=0  etf=1  h=2.52943e-07  ret=0
[cvode] pt=2/10  t=5.412e-06  steps=13  rhs=21  nlcf=0  etf=1  h=2.529e-07  ret=0
[cvode] pt=3/10  t=6.310e-06  steps=15  rhs=24  nlcf=0  etf=1  h=8.490e-07  ret=0
[cvode] pt=4/10  t=7.356e-06  steps=16  rhs=26  nlcf=0  etf=1  h=1.280e-06  ret=0
[cvode] pt=5/10  t=8.577e-06  steps=17  rhs=28  nlcf=0  etf=1  h=1.280e-06  ret=0
[cvode] pt=6/10  t=1.000e-05  steps=18  rhs=29  nlcf=0  etf=1  h=1.280e-06  ret=0
[cvode] pt=7/10  t=1.166e-05  steps=19  rhs=30  nlcf=0  etf=1  h=1.280e-06  ret=0
[cvode] pt=8/10  t=1.359e-05  steps=21  rhs=32  nlcf=0  etf=1  h=2.092e-06  ret=0
[cvode] pt=9/10  t=1.585e-05  steps=22  rhs=33  nlcf=0  etf=1  h=2.092e-06  ret=0
Done: 10 time points written.


C++ solver completed — 10 time points
Results processing complete.
  Tail OK: SIA=0.000  VAC=0.000

[Segment 3] I=10000  V=10000  t=[1.58e-05, 6.31e-05]

Launching solver_mode='cpp_full' …
C++ solver: solver.exe  N_eq=20006  solver_mode='cpp_full'  physics='bin_moment_CD_fission'


...

Expanded_Eurofer_CD solver: N_eq=20006  physics_option=2  he_mode=0  he_options=quasi_steady_state  C_floor=1e-25  window_mode=0
[cvode] pt=1/10  t=292864  steps=697  rhs=748  nlcf=2  etf=0  h=59.7484  ret=0
[cvode] pt=2/10  t=3.415e+05  steps=1510  rhs=1633  nlcf=2  etf=0  h=5.975e+01  ret=0
[cvode] pt=3/10  t=3.981e+05  steps=2458  rhs=2628  nlcf=2  etf=0  h=5.975e+01  ret=0
[cvode] pt=4/10  t=4.642e+05  steps=3217  rhs=3460  nlcf=2  etf=0  h=9.043e+01  ret=0


C++ solver failed (exit code 4294967295)
  Solver failed — returning accumulated results.
Diagnostics written to: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD\output\20260413_102957_cpp_full_bin_moment_CD_fission_5329808\diagnostics.txt


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeOneSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeTwoSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeThreeSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFourSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFiveSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmsy10'] not

Saved plots to C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD\output\20260413_102957_cpp_full_bin_moment_CD_fission_5329808\plots/
Output written to: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD\output\20260413_102957_cpp_full_bin_moment_CD_fission_5329808

Final domain: I=10000  V=10000
Solution:         172 time points, t = [1.00e-06, 2.51e+05] s
Final dose:       2.5119e-01 dpa
Swelling (final): 0.744753 %
C_He_tot (final): 2.617e+20 m^-3
mean_n_i (final): 19.65
mean_n_v (final): 886.08
N_loops  (final): 1.508e+24 m^-3
N_voids  (final): 7.123e+23 m^-3
delta_FP (final): 1.032e-01  (Eq. 164)
delta_He (final): 2.500e-01  (Eq. 165)


c:\Users\Owner\Documents\Repos\EuroferMicrostructure\.EuroVenv\Lib\site-packages\IPython\core\events.py:100: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  func(*args, **kwargs)


## Recover partial results after interrupt

If you interrupted the simulation cell above (Kernel → Interrupt or `Ctrl+C`),
run the cell below to recover all completed segments, generate plots, and save output.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# RECOVER PARTIAL RESULTS after KeyboardInterrupt
# ══════════════════════════════════════════════════════════════════════════════
# If `results` was already set by a graceful interrupt (the modified
# run_adaptive catches KeyboardInterrupt and returns accumulated data),
# use it directly.  Otherwise fall back to sim._accumulated_results.

partial = results if (results is not None) else getattr(sim, '_accumulated_results', None)

if partial is not None and partial['t'].size > 0:
    results = partial
    t_arr   = results['t']
    print(f'Recovered {len(t_arr)} time points,  '
          f't = [{t_arr[0]:.2e}, {t_arr[-1]:.2e}] s')
    print(f'Final dose:       {results["dose"][-1]:.4e} dpa')
    print(f'Swelling (final): {results["swelling"][-1]*100:.6f} %')
    print(f'C_He_tot (final): {results["C_He_tot"][-1]:.3e} m^-3')
    print(f'mean_n_i (final): {results["mean_n_i"][-1]:.2f}')
    print(f'mean_n_v (final): {results["mean_n_v"][-1]:.2f}')
    print(f'N_loops  (final): {results["N_loops"][-1]:.3e} m^-3')
    print(f'N_voids  (final): {results["N_voids"][-1]:.3e} m^-3')
    print(f'delta_FP (final): {results["delta_FP"][-1]:.3e}')
    print(f'delta_He (final): {results["delta_He"][-1]:.3e}')

    # ── Save plots ────────────────────────────────────────────────────────
    from datetime import datetime
    out_dir   = MODULE_ROOT / 'output' / f'{datetime.now():%Y%m%d_%H%M%S}_partial'
    plots_dir = out_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    viz.save_all_plots(results, sim.input_data, str(plots_dir),
                       label='partial', rate_eq_obj=sim.rate_equations)
    print(f'\nPlots saved to:  {plots_dir}')

    # ── Save raw arrays ──────────────────────────────────────────────────
    np.save(str(out_dir / 'results_t.npy'), results['t'])
    np.save(str(out_dir / 'results_y.npy'), results['y'])
    print(f'Raw data saved to: {out_dir}')
else:
    print('No partial results available.')
    print('Either the first segment has not completed yet, or the sim object')
    print('was not created.  Wait for at least one segment to finish before')
    print('interrupting.')

NameError: name 'results' is not defined